# 18. Project: AI Interviewer

Time to build something real! We will make an **AI Interviewer** that asks you
job-interview questions **one at a time**, waits for your answer, and at the end
gives you **friendly feedback**.

This project uses ideas from the whole course: an **agent**, a **team**,
**human in the loop**, and **termination**.

## What we will build

| Member | Role |
|--------|------|
| **interviewer** (AI) | Asks one interview question at a time, then gives feedback |
| **you** (`UserProxyAgent`) | Answer each question by typing |

The interview **stops** when the interviewer says **"END OF INTERVIEW"**
(after a few questions and feedback).

## How it flows

```
interviewer: asks question 1
you:         type your answer
interviewer: asks question 2
you:         type your answer
...
interviewer: gives feedback + "END OF INTERVIEW"   -> stops
```

This is a **RoundRobin team** of the interviewer and you, with a **stop word**.

In [1]:
from dotenv import load_dotenv
load_dotenv()

from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

# The AI interviewer with clear instructions (prompt engineering!)
interviewer = AssistantAgent(
    name="interviewer",
    model_client=model_client,
    system_message=(
        "You are a friendly job interviewer for a beginner Python developer role. "
        "Ask ONE short question at a time and wait for the answer. "
        "Ask 3 questions in total. After the 3rd answer, give short, kind feedback "
        "and finish with the exact words: END OF INTERVIEW."
    ),
)

# You answer the questions
candidate = UserProxyAgent(name="candidate")

# Stop when the interviewer says END OF INTERVIEW
interview = RoundRobinGroupChat(
    [interviewer, candidate],
    termination_condition=TextMentionTermination("END OF INTERVIEW"),
)

# Start the interview. Type your answers when the box appears.
await Console(interview.run_stream(task="Please start the interview."))

---------- TextMessage (user) ----------
Please start the interview.
---------- TextMessage (interviewer) ----------
Sure! Let’s get started. 

First question: Can you briefly explain what Python is and why it’s popular for beginners?
---------- TextMessage (candidate) ----------
python is coding language
---------- TextMessage (interviewer) ----------
That's a good start! Python is indeed a programming language. It’s popular among beginners due to its readability and simplicity. 

Second question: What is a variable in Python, and how do you create one?
---------- TextMessage (candidate) ----------

---------- TextMessage (interviewer) ----------
It seems like you may not have provided an answer. That's okay! 

Let’s move on to the third question: Can you tell me what a list is in Python and give an example of how to create one?
---------- TextMessage (candidate) ----------
type
---------- TextMessage (interviewer) ----------
Thank you for your responses! Just a little feedback: it's 

TaskResult(messages=[TextMessage(id='c95455f4-3c99-4255-b598-1e6647e7aafb', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 7, 7, 12, 47, 22, 970582, tzinfo=datetime.timezone.utc), content='Please start the interview.', type='TextMessage'), TextMessage(id='b23f781d-248d-4af9-9802-14ae45e211d6', source='interviewer', models_usage=RequestUsage(prompt_tokens=74, completion_tokens=26), metadata={}, created_at=datetime.datetime(2026, 7, 7, 12, 47, 26, 765393, tzinfo=datetime.timezone.utc), content='Sure! Let’s get started. \n\nFirst question: Can you briefly explain what Python is and why it’s popular for beginners?', type='TextMessage'), UserInputRequestedEvent(id='c0f7c09d-dbe4-4eff-a48a-c24cedecce3f', source='candidate', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 7, 7, 12, 47, 26, 768993, tzinfo=datetime.timezone.utc), request_id='bd7c20c0-f4ff-42f4-9c51-3c897fe4cf77', content='', type='UserInputRequestedEvent'), TextMessage(id='e

## Try it yourself

- Change the **role** (e.g. "data analyst" instead of "Python developer").
- Change the **number of questions** (ask 5 instead of 3).
- Make the interviewer **stricter** or **friendlier** by editing the `system_message`.

## Key points to remember

- A **real project** combines many ideas: agent + team + human-in-the-loop + termination.
- A clear **`system_message`** controls the whole behaviour (ask one at a time, 3 questions, end word).
- **`UserProxyAgent`** lets **you** answer each question.
- **`TextMentionTermination`** ends the interview at the right moment.
- You can easily **customize** the project by editing the instructions.